In [ ]:
import pandas as pd
import datetime
import numpy as np
from copy import deepcopy
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from catboost import CatBoostClassifier, Pool

In [ ]:
def uplift_fit_predict(model, X_train, treatment_train, target_train, X_test):
    """
    Реализация простого способа построения uplift-модели.
    
    Обучаем два бинарных классификатора, которые оценивают вероятность target для клиента:
    1. с которым была произведена коммуникация (treatment=1)
    2. с которым не было коммуникации (treatment=0)
    
    В качестве оценки uplift для нового клиента берется разница оценок вероятностей:
    Predicted Uplift = P(target|treatment=1) - P(target|treatment=0)
    """
    X_treatment, y_treatment = X_train[treatment_train == 1, :], target_train[treatment_train == 1]
    X_control, y_control = X_train[treatment_train == 0, :], target_train[treatment_train == 0]
    model_treatment = clone(model).fit(X_treatment, y_treatment)
    model_control = clone(model).fit(X_control, y_control)
    predict_treatment = model_treatment.predict_proba(X_test)[:, 1]
    predict_control = model_control.predict_proba(X_test)[:, 1]
    predict_uplift = predict_treatment - predict_control
    return predict_uplift

def uplift_fit_predict_catboost(model, X_train, treatment_train, target_train, X_test, cat_features):
    """
    :param model: 
    :param X_train: 
    :param treatment_train: 
    :param target_train: 
    :param X_test: 
    :return: 
    """
    X_treatment = Pool(X_train[treatment_train == 1], target_train[treatment_train == 1], cat_features=cat_features)
    X_control = Pool(X_train[treatment_train == 0], target_train[treatment_train == 0], cat_features=cat_features)
    model_treatment = deepcopy(model).fit(X_treatment, verbose=200, early_stopping_rounds=200)
    model_control = deepcopy(model).fit(X_control, verbose=200, early_stopping_rounds=200)
    predict_treatment = model_treatment.predict_proba(X_test)[:, 1]
    predict_control = model_control.predict_proba(X_test)[:, 1]
    predict_uplift = predict_treatment - predict_control
    return predict_uplift



def uplift_score(prediction, treatment, target, rate=0.3):
    """
    Подсчет Uplift Score
    """
    order = np.argsort(-prediction)
    treatment_n = int((treatment == 1).sum() * rate)
    treatment_p = target[order][treatment[order] == 1][:treatment_n].mean()
    control_n = int((treatment == 0).sum() * rate)
    control_p = target[order][treatment[order] == 0][:control_n].mean()
    score = treatment_p - control_p
    return score

In [58]:
df_train = pd.read_csv('./data/data/uplift_train.csv', index_col='client_id')
df_test = pd.read_csv('./data/data/uplift_test.csv', index_col='client_id')
dataset = pd.read_parquet('./data/preprocessed/dataset_v0.parquet')
df_train = df_train.merge(
    dataset,
    left_index=True,
    right_index=True,
    how='left'
)
df_test = df_test.merge(
    dataset,
    left_index=True,
    right_index=True,
    how='left'
)
cat_features = ['gender', 'age_oon', 'issue_redeem_delay_oon']

In [93]:
model = CatBoostClassifier(
    loss_function='Logloss',
    eval_metric='AUC',
    learning_rate=0.05,
    iterations=400,
    depth=4,
    random_strength=0,
    l2_leaf_reg=0.5,
    task_type='GPU',
    random_seed=42,
    verbose=True
)

In [94]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 200039 entries, 000012768d to fffff6ce77
Data columns (total 32 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   treatment_flg                 200039 non-null  int64  
 1   target                        200039 non-null  int64  
 2   age                           200039 non-null  int64  
 3   gender                        200039 non-null  object 
 4   first_issue_unixtime          200039 non-null  float64
 5   first_redeem_unixtime         200039 non-null  float64
 6   issue_redeem_delay            200039 non-null  float64
 7   age_oon                       200039 non-null  bool   
 8   issue_redeem_delay_oon        200039 non-null  bool   
 9   transaction_id_nunique        200039 non-null  int64  
 10  regular_points_received_sum   200039 non-null  float64
 11  regular_points_received_mean  200039 non-null  float64
 12  express_points_received_sum   200039

In [95]:
indices_train = df_train.index
indices_test = df_test.index
indices_learn, indices_valid = train_test_split(df_train.index, test_size=0.3, random_state=123)

valid_uplift = uplift_fit_predict_catboost(
    model=model,
    X_train=df_train.loc[indices_learn, :].drop(columns=['treatment_flg', 'target']),
    treatment_train=df_train.loc[indices_learn, 'treatment_flg'],
    target_train=df_train.loc[indices_learn, 'target'],
    X_test=df_train.loc[indices_valid, :].drop(columns=['treatment_flg', 'target']),
    cat_features=cat_features
)
valid_score = uplift_score(
    valid_uplift,
    treatment=df_train.loc[indices_valid, 'treatment_flg'].values,
    target=df_train.loc[indices_valid, 'target'].values,
)
print('Validation score:', valid_score)

Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 8.66ms	remaining: 3.46s
200:	total: 1.69s	remaining: 1.68s
399:	total: 3.29s	remaining: 0us


Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 7.98ms	remaining: 3.18s
200:	total: 1.53s	remaining: 1.52s
399:	total: 3.11s	remaining: 0us
Validation score: 0.0645431130849875


In [98]:
test_uplift = uplift_fit_predict_catboost(
    model=model,
    X_train=df_train.loc[indices_learn, :].drop(columns=['treatment_flg', 'target']),
    treatment_train=df_train.loc[indices_learn, 'treatment_flg'],
    target_train=df_train.loc[indices_learn, 'target'],
    X_test=df_test.loc[indices_test, :].drop(columns=['treatment_flg', 'target'], errors='ignore'),
    cat_features=cat_features
)

df_submission = pd.DataFrame({'uplift': test_uplift}, index=df_test.index)
df_submission.to_csv('./data/submission.csv')

Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 9.57ms	remaining: 3.82s
200:	total: 1.72s	remaining: 1.7s
399:	total: 3.29s	remaining: 0us


Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 9.14ms	remaining: 3.65s
200:	total: 1.57s	remaining: 1.55s
399:	total: 3.11s	remaining: 0us
